# Modern Attention Mechanisms

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/transformers/04-modern-attention

A runnable tour of cross-attention shapes, KV cache size trade-offs (MHA / GQA / MQA),
RoPE rotation (pure math verification), FlashAttention memory complexity, and
sliding-window sparsity — all in pure NumPy / stdlib so every result is deterministic.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = '#e2e8f0'
plt.rcParams['axes.labelcolor']  = '#e2e8f0'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#334155'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.color']       = '#1e293b'
plt.rcParams['figure.figsize']   = (9, 5)
np.random.seed(0)

## 1 — Cross-attention: tensor shapes

In encoder–decoder cross-attention the **queries** come from the decoder and the
**keys/values** come from the encoder output.  The two sequences can have different
lengths — that is the whole point.

$$Q = X_{\text{dec}}\,W_Q,\qquad K = X_{\text{enc}}\,W_K,\qquad V = X_{\text{enc}}\,W_V$$
$$\text{CrossAttn} = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

The score matrix $QK^\top$ has shape $(T_{\text{dec}} \times T_{\text{enc}})$ —
each decoder position scores every encoder position.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.RandomState(1)

d_model = 32
d_k     = 16          # projection dimension
T_enc   = 6           # source tokens (e.g. "Hola mundo <EOS>" + padding)
T_dec   = 4           # generated tokens so far

X_enc = rng.randn(T_enc, d_model)   # encoder output
X_dec = rng.randn(T_dec, d_model)   # decoder hidden states

Wq = rng.randn(d_model, d_k) * 0.1
Wk = rng.randn(d_model, d_k) * 0.1
Wv = rng.randn(d_model, d_k) * 0.1

Q = X_dec @ Wq          # (T_dec, d_k)
K = X_enc @ Wk          # (T_enc, d_k)
V = X_enc @ Wv          # (T_enc, d_k)

scores = Q @ K.T / math.sqrt(d_k)   # (T_dec, T_enc)
weights = softmax(scores)            # (T_dec, T_enc)  — rows sum to 1
context = weights @ V                # (T_dec, d_k)

print(f"Q shape:       {Q.shape}   (T_dec × d_k)")
print(f"K shape:       {K.shape}   (T_enc × d_k)")
print(f"score matrix:  {scores.shape}  (T_dec × T_enc)")
print(f"context:       {context.shape}  (T_dec × d_k)")

# Visualise the attention weights for each decoder position
fig, ax = plt.subplots(figsize=(7, 3.5))
im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=weights.max())
ax.set_xlabel('Encoder position'); ax.set_ylabel('Decoder position')
ax.set_title('Cross-attention weights  (T_dec × T_enc)')
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## 2 — KV cache size: MHA vs GQA vs MQA

At inference each new token generation step requires reading the keys and values
for **every previous token**.  The KV cache stores these to avoid recomputation.

| Variant | KV heads | Cache formula |
|---------|----------|---------------|
| MHA | $h$ | $2 \times N \times h \times d_k \times$ bytes |
| GQA | $G$ | $2 \times N \times G \times d_k \times$ bytes |
| MQA | $1$ | $2 \times N \times 1 \times d_k \times$ bytes |

Below we plot cache size (MB, bf16) vs sequence length for a LLaMA-2-style config
($h=32$, $d_k=128$, 32 layers) under MHA, GQA ($G=8$), and MQA.

In [ ]:
h        = 32      # query heads
d_k      = 128     # head dimension
n_layers = 32
bytes_per_elem = 2  # bfloat16

seq_lens = np.arange(256, 8193, 256)

def kv_cache_mb(N, n_kv_heads):
    """Total KV cache in MB across all layers."""
    per_layer = 2 * N * n_kv_heads * d_k * bytes_per_elem
    return per_layer * n_layers / (1024 ** 2)

mha_mb  = [kv_cache_mb(N, h)   for N in seq_lens]
gqa_mb  = [kv_cache_mb(N, 8)   for N in seq_lens]   # G=8
mqa_mb  = [kv_cache_mb(N, 1)   for N in seq_lens]

plt.plot(seq_lens, mha_mb, label='MHA (32 KV heads)',  color='#6366f1')
plt.plot(seq_lens, gqa_mb, label='GQA G=8 (8 KV heads)', color='#14b8a6', linestyle='--')
plt.plot(seq_lens, mqa_mb, label='MQA (1 KV head)',    color='#f59e0b', linestyle=':')
plt.xlabel('Sequence length (tokens)')
plt.ylabel('KV cache (MB, all layers)')
plt.title('KV cache size vs sequence length  |  LLaMA-2 config')
plt.legend()
plt.tight_layout(); plt.show()

N_test = 4096
print(f"At N={N_test} tokens:")
print(f"  MHA  : {kv_cache_mb(N_test, h):.1f} MB")
print(f"  GQA-8: {kv_cache_mb(N_test, 8):.1f} MB  ({h//8}× reduction)")
print(f"  MQA  : {kv_cache_mb(N_test, 1):.1f} MB  ({h}× reduction)")

## 3 — RoPE: rotation and relative-distance verification

RoPE encodes position $m$ by rotating each pair of dimensions $(q_{2i},\,q_{2i+1})$
by angle $m\,\theta_i$ where $\theta_i = \text{base}^{-2i/d}$.

$$
\text{RoPE}(\mathbf{q}, m)_{2i,\,2i+1}
= \begin{pmatrix}\cos m\theta_i & -\sin m\theta_i \\
                  \sin m\theta_i &  \cos m\theta_i\end{pmatrix}
  \begin{pmatrix}q_{2i}\\ q_{2i+1}\end{pmatrix}
$$

Key property: $\text{RoPE}(\mathbf{q},m)^\top \cdot \text{RoPE}(\mathbf{k},n)$
depends only on $m - n$ (relative distance), not on $m$ and $n$ separately.

In [ ]:
def rope_apply(v, pos, base=10000):
    """Apply RoPE rotation to vector v at position pos (pure stdlib/numpy)."""
    d = len(v)
    assert d % 2 == 0, "d must be even"
    out = np.empty_like(v)
    for i in range(d // 2):
        theta_i = base ** (-2 * i / d)
        cos_t   = math.cos(pos * theta_i)
        sin_t   = math.sin(pos * theta_i)
        out[2*i]   = cos_t * v[2*i]   - sin_t * v[2*i+1]
        out[2*i+1] = sin_t * v[2*i]   + cos_t * v[2*i+1]
    return out

rng2 = np.random.RandomState(42)
d = 8
q = rng2.randn(d)
k = rng2.randn(d)

# Dot products for several (pos_q, pos_k) pairs with the same relative offset = 3
relative_offset = 3
pairs = [(0, 3), (5, 8), (10, 13), (20, 23)]
dots = []
for pq, pk in pairs:
    q_rot = rope_apply(q, pq)
    k_rot = rope_apply(k, pk)
    dots.append(float(np.dot(q_rot, k_rot)))

print("Relative-distance property verification")
print(f"All pairs have pos_q - pos_k = {relative_offset}")
for (pq, pk), dot in zip(pairs, dots):
    print(f"  pos_q={pq:2d}, pos_k={pk:2d}  →  dot = {dot:.6f}")

# All dots should be equal (same relative offset ⟹ same dot product)
assert max(dots) - min(dots) < 1e-10, "RoPE relative-distance property violated!"
print("Relative-distance check: PASS  (all dots identical)")

# Also verify that different offsets give different dots
dots_diff = []
for offset in [0, 1, 2, 3, 4, 5]:
    q_rot = rope_apply(q, 0)
    k_rot = rope_apply(k, offset)
    dots_diff.append(float(np.dot(q_rot, k_rot)))

plt.figure(figsize=(7, 3))
plt.bar(range(6), dots_diff, color='#6366f1')
plt.xlabel('Relative offset (pos_q=0, pos_k=offset)')
plt.ylabel('Attention dot product')
plt.title('RoPE: attention score changes with relative offset')
plt.tight_layout(); plt.show()

## 4 — FlashAttention: memory complexity comparison

Standard attention materialises the full $N\times N$ score matrix in HBM (slow GPU memory).
FlashAttention tiles the computation into blocks that fit in SRAM and never writes the full
matrix — memory scales as $O(N)$ instead of $O(N^2)$.  The math is **exact** (no approximation).

Below we compare the theoretical memory footprint (float16, single head) as a function
of sequence length.

In [ ]:
d_k       = 64      # head dimension
bytes_f16 = 2       # float16

seq_lens_flash = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536])

# Standard: N×N score matrix + N×N softmax output (dominant term)
standard_mb = 2 * seq_lens_flash**2 * bytes_f16 / (1024**2)

# FlashAttention: O(N) — store Q, K, V plus small per-block accumulators
# Dominant term: 3 × N × d_k (Q, K, V buffers) plus O(N) running stats
flash_mb = 3 * seq_lens_flash * d_k * bytes_f16 / (1024**2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale
axes[0].plot(seq_lens_flash, standard_mb, 'o-', label='Standard O(N²)', color='#f43f5e')
axes[0].plot(seq_lens_flash, flash_mb,    's-', label='Flash   O(N)',    color='#14b8a6')
axes[0].set_xlabel('Sequence length'); axes[0].set_ylabel('HBM memory (MB, 1 head)')
axes[0].set_title('Linear scale'); axes[0].legend()

# Log scale — shows the divergence clearly
axes[1].semilogy(seq_lens_flash, standard_mb, 'o-', label='Standard O(N²)', color='#f43f5e')
axes[1].semilogy(seq_lens_flash, flash_mb,    's-', label='Flash   O(N)',    color='#14b8a6')
axes[1].set_xlabel('Sequence length'); axes[1].set_ylabel('HBM memory (MB, log scale)')
axes[1].set_title('Log scale'); axes[1].legend()

plt.suptitle('FlashAttention memory footprint vs standard attention  (single head, d_k=64)')
plt.tight_layout(); plt.show()

print(f"At N=32768:  standard = {standard_mb[-2]:.0f} MB,  flash = {flash_mb[-2]:.1f} MB")
print(f"Memory ratio (standard/flash): {standard_mb[-2]/flash_mb[-2]:.0f}×")

## 5 — Sliding-window attention

Full attention is $O(N^2)$.  Sliding-window attention limits each token to attending
within a window of size $w$ — complexity drops to $O(N \cdot w)$.

We visualise the attention mask for both patterns and compare the number of active
entries (proportional to compute and memory).

In [ ]:
N = 24     # token sequence length (small, for visualisation)
w = 6      # window size (each token attends to w/2 neighbours on each side)

# Full causal mask  (lower triangular)
full_mask = np.tril(np.ones((N, N)))

# Sliding-window causal mask
sliding_mask = np.zeros((N, N))
for i in range(N):
    lo = max(0, i - w // 2)
    hi = i + 1                # causal: only attend to past + current
    sliding_mask[i, lo:hi] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mask, title in zip(axes,
                            [full_mask, sliding_mask],
                            ['Full causal attention  O(N²)',
                             f'Sliding-window (w={w}) causal  O(N·w)']):
    ax.imshow(mask, cmap='Blues', vmin=0, vmax=1, origin='upper')
    ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
    ax.set_title(title)
plt.tight_layout(); plt.show()

active_full    = int(full_mask.sum())
active_sliding = int(sliding_mask.sum())
print(f"Active attention pairs — full: {active_full},  sliding-window: {active_sliding}")
print(f"Reduction: {active_full / active_sliding:.1f}×  (for N={N}, w={w})")

# Complexity scaling
Ns = np.array([1024, 4096, 16384, 65536, 131072])
w_fixed = 4096
full_ops    = Ns**2
sliding_ops = Ns * w_fixed
print("\nCompute comparison (relative ops):")
for N_val, f, s in zip(Ns, full_ops, sliding_ops):
    print(f"  N={N_val:7d}: full={f:>14,.0f}  sliding={s:>14,.0f}  ratio={f/s:.0f}×")

## Key takeaways

- **Cross-attention**: Q from decoder, K/V from encoder; score matrix is $(T_{\text{dec}} \times T_{\text{enc}})$.
- **GQA/MQA** cut KV cache size by sharing K/V heads across query head groups; MQA ($G=1$) is the extreme case.
- **RoPE** rotates Q and K at their positions; the dot product then depends only on relative offset $m-n$.
- **FlashAttention** tiles Q/K/V to avoid the $O(N^2)$ HBM allocation; memory becomes $O(N)$ with no approximation.
- **Sliding-window** attention reduces compute from $O(N^2)$ to $O(N\cdot w)$; global tokens preserve long-range connectivity.

---
## ✏️ Your turn

### Exercise 1 — RoPE rotation

Implement the RoPE rotation of a query vector, then verify the **relative-distance property**:
if two pairs $(m_1, n_1)$ and $(m_2, n_2)$ have the same relative offset
$m_1 - n_1 = m_2 - n_2$, their rotated dot products must be identical.

$$
\text{RoPE}(\mathbf{q}, m)_{2i,\,2i+1}
= \begin{pmatrix}\cos m\theta_i & -\sin m\theta_i \\
                  \sin m\theta_i &  \cos m\theta_i\end{pmatrix}
  \begin{pmatrix}q_{2i}\\ q_{2i+1}\end{pmatrix},
\qquad \theta_i = 10000^{-2i/d}
$$

In [ ]:
import math
import numpy as np

def rope_rotate(q, pos, base=10000):
    """
    Apply RoPE rotation to 1-D vector q at sequence position pos.

    Args:
        q    : 1-D numpy array of length d (d must be even)
        pos  : integer, the token's position in the sequence
        base : RoPE base (default 10000)

    Returns:
        1-D numpy array of the same shape as q, rotated.
    """
    # TODO(you): for each dimension pair (2i, 2i+1):
    #   compute theta_i = base ** (-2*i / len(q))
    #   rotate (q[2i], q[2i+1]) by angle (pos * theta_i)
    ...

In [ ]:
rng_ex1 = np.random.RandomState(7)
d = 8
q_ex = rng_ex1.randn(d)
k_ex = rng_ex1.randn(d)

# Shape check
q_rot_0 = rope_rotate(q_ex, 0)
assert q_rot_0.shape == q_ex.shape, \
    f"rope_rotate must return same shape as input, got {q_rot_0.shape}"

# At position 0 all angles are 0, so RoPE(q, 0) == q
assert np.allclose(q_rot_0, q_ex, atol=1e-8), \
    "RoPE at position 0 must leave the vector unchanged (all angles = 0)"

# Relative-distance property:
# pairs (pos_q, pos_k) all have the same relative offset = 5
offset = 5
test_pairs = [(0, 5), (3, 8), (10, 15), (100, 105)]
dots_ex = []
for pq, pk in test_pairs:
    q_r = rope_rotate(q_ex, pq)
    k_r = rope_rotate(k_ex, pk)
    dots_ex.append(float(np.dot(q_r, k_r)))

assert max(dots_ex) - min(dots_ex) < 1e-10, (
    f"Relative-distance property failed.\n"
    f"All pairs have offset {offset} but got dots: {dots_ex}"
)

# Different offsets must give different dot products (sanity check)
dot_offset_2 = np.dot(rope_rotate(q_ex, 0), rope_rotate(k_ex, 2))
dot_offset_5 = np.dot(rope_rotate(q_ex, 0), rope_rotate(k_ex, 5))
assert not math.isclose(dot_offset_2, dot_offset_5, abs_tol=1e-8), \
    "Different offsets must produce different dot products"


# Edge case: d = 2 (smallest even dimension -- exactly one rotation pair, theta_0 = 1)
q2 = np.array([1.0, 0.0])
rotated = rope_rotate(q2, pos=1)
assert np.allclose(rotated, [math.cos(1.0), math.sin(1.0)], atol=1e-9), \
    "with d=2 there is exactly one frequency (theta_0=1); rotating [1,0] by pos=1 rad"

# Edge case: the zero vector rotates to the zero vector at any position
assert np.allclose(rope_rotate(np.zeros(8), 37), np.zeros(8)), \
    "rotating the zero vector must yield the zero vector at any position"

print("✅ Exercise 1 passed — rope_rotate is correct")

<details>
<summary>💡 Show solution</summary>

```python
def rope_rotate(q, pos, base=10000):
    d   = len(q)
    out = np.empty_like(q)
    for i in range(d // 2):
        theta_i = base ** (-2 * i / d)
        cos_t   = math.cos(pos * theta_i)
        sin_t   = math.sin(pos * theta_i)
        out[2*i]   = cos_t * q[2*i]   - sin_t * q[2*i+1]
        out[2*i+1] = sin_t * q[2*i]   + cos_t * q[2*i+1]
    return out
```

</details>

### Exercise 2 — GQA KV cache size

Implement a function that computes the KV cache size (in **number of scalar elements**,
not bytes) for a single layer, then verify the three special cases:

| Variant | `n_kv_heads` | expected cache elements |
|---------|-------------|-------------------------|
| MHA | `n_heads` | $2 \times N \times n_{\text{heads}} \times d_k$ |
| MQA | `1` | $2 \times N \times 1 \times d_k$ |
| GQA | `G` | $2 \times N \times G \times d_k$ |

The factor of 2 accounts for storing **both** K and V.

In [ ]:
def kv_cache_size(seq_len, n_kv_heads, d_head):
    """
    Return the number of scalar elements in the KV cache for ONE layer.

    Args:
        seq_len    : int, current sequence length N
        n_kv_heads : int, number of KV heads (G for GQA; 1 for MQA; n_heads for MHA)
        d_head     : int, dimension per head (d_k)

    Returns:
        int, total number of scalar values in the KV cache for this layer
    """
    # TODO(you): 2 tensors (K and V), each of shape (seq_len, n_kv_heads, d_head)
    ...

In [ ]:
N      = 1024
H      = 32    # total query heads
d_head = 128

# MHA: n_kv_heads == n_heads
size_mha = kv_cache_size(N, H, d_head)
assert size_mha == 2 * N * H * d_head, (
    f"MHA cache should be {2*N*H*d_head}, got {size_mha}"
)

# MQA: n_kv_heads == 1
size_mqa = kv_cache_size(N, 1, d_head)
assert size_mqa == 2 * N * 1 * d_head, (
    f"MQA cache should be {2*N*1*d_head}, got {size_mqa}"
)

# GQA with G=8
G = 8
size_gqa = kv_cache_size(N, G, d_head)
assert size_gqa == 2 * N * G * d_head, (
    f"GQA-8 cache should be {2*N*G*d_head}, got {size_gqa}"
)

# Reduction factors relative to MHA
assert size_mha // size_mqa == H, \
    f"MQA should be {H}× smaller than MHA"
assert size_mha // size_gqa == H // G, \
    f"GQA-{G} should be {H//G}× smaller than MHA"


# Edge case: seq_len = 1 (single-token sequence / first generation step)
assert kv_cache_size(1, H, d_head) == 2 * 1 * H * d_head

# Edge case: n_kv_heads == 0 would mean no cache at all
assert kv_cache_size(N, 0, d_head) == 0

print("✅ Exercise 2 passed — kv_cache_size is correct")
print(f"   MHA  : {size_mha:,} elements")
print(f"   GQA-8: {size_gqa:,} elements  ({size_mha//size_gqa}× smaller)")
print(f"   MQA  : {size_mqa:,} elements  ({size_mha//size_mqa}× smaller)")

<details>
<summary>💡 Show solution</summary>

```python
def kv_cache_size(seq_len, n_kv_heads, d_head):
    # K tensor: (seq_len, n_kv_heads, d_head)
    # V tensor: (seq_len, n_kv_heads, d_head)
    # Total = 2 × seq_len × n_kv_heads × d_head
    return 2 * seq_len * n_kv_heads * d_head
```

</details>

### Exercise 3 — DML practice: `sparse_window_attention` (DML #131)

Sliding-window attention, as an exact function: for each token `i`, attend only to keys in
`[max(0, i - window_size), min(seq_len, i + window_size + 1))` -- a **symmetric** window
(unlike Section 5 above, whose visualisation used a *causal*, backward-only window).
`scale_factor` defaults to $\sqrt{d_k}$ when not given.

In [ ]:
def sparse_window_attention(Q, K, V, window_size, scale_factor=None):
    """DML #131: attend each query only to keys within `window_size` positions
    on either side (no causal restriction -- a genuinely symmetric window).

    Args:
        Q, K: (seq_len, d_k);  V: (seq_len, d_v)
        window_size: int radius (each token attends to window_size positions on each side)
        scale_factor: defaults to sqrt(d_k) if None
    Returns:
        (seq_len, d_v) attention output
    """
    # TODO(you): loop over query positions i, slice K/V to
    #   [max(0, i-window_size), min(seq_len, i+window_size+1)), then softmax-attend
    ...

In [ ]:
import numpy as np

# DML #131 -- exact test vectors from tests.json
Q = np.array([[1.0], [1.0], [1.0]]); K = Q.copy(); V = np.array([[1.0], [2.0], [3.0]])
assert np.allclose(sparse_window_attention(Q, K, V, 1), [[1.5], [2.0], [2.5]], atol=1e-6)

# Edge case: single-token sequence with window_size=0 -- attends only to itself
Q1 = np.array([[4.0]]); K1 = np.array([[4.0]]); V1 = np.array([[5.0]])
assert np.allclose(sparse_window_attention(Q1, K1, V1, 0), [[5.0]], atol=1e-6)

Q3 = np.array([[0.0], [1.0], [0.0], [2.0], [0.0], [7.0]])
K3 = np.array([[1.0], [2.0], [3.0], [0.0], [6.0], [0.0]])
V3 = np.array([[10.0], [20.0], [30.0], [12.0], [23.0], [70.0]])
expected3 = [[20.0], [25.31123059], [19.0], [23.01651938], [33.75], [23.0]]
assert np.allclose(sparse_window_attention(Q3, K3, V3, 2), expected3, atol=1e-5)

# Edge case: window_size >= seq_len degenerates to full (dense) attention
Qd = np.random.RandomState(5).randn(4, 3); Kd = Qd.copy(); Vd = np.random.RandomState(6).randn(4, 3)
full = sparse_window_attention(Qd, Kd, Vd, window_size=100)

def dense_attention(Q, K, V):
    s = Q @ K.T / np.sqrt(Q.shape[1])
    w = np.exp(s - s.max(axis=1, keepdims=True)); w /= w.sum(axis=1, keepdims=True)
    return w @ V

assert np.allclose(full, dense_attention(Qd, Kd, Vd), atol=1e-6), \
    "a window covering the whole sequence must match dense (unwindowed) attention"

print("✅ Exercise 3 passed (DML #131 sparse window attention)")

<details>
<summary>💡 Show solution</summary>

```python
def sparse_window_attention(Q, K, V, window_size, scale_factor=None):
    seq_len, d_k = Q.shape
    if scale_factor is None:
        scale_factor = np.sqrt(d_k)
    output = np.zeros((seq_len, V.shape[1]))
    for i in range(seq_len):
        lo = max(0, i - window_size)
        hi = min(seq_len, i + window_size + 1)
        scores = Q[i:i+1] @ K[lo:hi].T / scale_factor
        scores = scores - scores.max(axis=1, keepdims=True)
        weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)
        output[i] = weights @ V[lo:hi]
    return output
```

</details>